In [5]:
partition = 200

In [6]:
import sys
from train import main
from itertools import product  
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [7]:
import re

def load_tested_configs(log_path):
    tested = set()
    with open(log_path, 'r') as f:
        for line in f:
            if line.startswith("Running:"):
                match = re.findall(r"[-\w.]+=\S+", line)
                if match:
                    # Normalize values to correct types
                    config = tuple([
                        int(re.search(r"=(\d+)", match[0]).group(1)),       # n_tree
                        int(re.search(r"=(\d+)", match[1]).group(1)),       # t_depth
                        int(re.search(r"=(\d+)", match[2]).group(1)),       # hd
                        int(re.search(r"=(\d+)", match[3]).group(1)),       # batch_size
                        float(re.search(r"=(\d+\.?\d*)", match[4]).group(1)), # feature_rate
                        float(re.search(r"=(\d+\.?\d*)", match[5]).group(1)), # dropout
                        float(re.search(r"=(\d+\.?\d*)", match[6]).group(1)), # lr
                    ])
                    tested.add(config)
    return tested


In [8]:
import random
from itertools import product
import sys

log_path = f"logs{partition}.txt"
#tested_configs = load_tested_configs(log_path)

n_tree_values = [5, 10, 20, 50, 100]
tree_depth_values = [6, 7, 8, 9, 10]
hidden_dim = [1024, 768]
batch_size_values = [256, 512]
tree_feature_rates = [0.0, 0.1, 0.2, 0.3, 0.4]
feat_dropouts = [0.0, 0.1, 0.2, 0.3]
lrs = [0.001, 0.01]

n_iter = 50
best_score = 0
best_config = {}

param_space = list(product(
    n_tree_values,
    tree_depth_values,
    hidden_dim,
    batch_size_values,
    tree_feature_rates,
    feat_dropouts,
    lrs
))

best_acc = 0

#available_configs = [cfg for cfg in param_space if cfg not in tested_configs]
available_configs = [cfg for cfg in param_space]
sampled_configs = random.sample(available_configs, min(n_iter, len(available_configs)))
i = 1

for n_tree, t_depth, hd, batch_size, feature_rate, dropout, lr in sampled_configs:
    log_line = f"Running: n_tree={n_tree}, t_depth={t_depth}, hd={hd}, batch_size={batch_size}, feature_rate={feature_rate}, dropout={dropout}, lr={lr}"
    print(f"\n{log_line}")
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{log_line}\n")

    sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(n_tree),
        '-tree_depth', str(t_depth),
        '-batch_size', str(batch_size),
        '-hidden_dim', str(hd),
        '-tree_feature_rate', str(feature_rate),
        '-feat_dropout', str(dropout),
        '-lr', str(lr),
        '-epochs', '400',
        '-verbose', '0',
        '-jointly_training',
        '-searching', '1'
    ]

    print(f"{i} / 100")
    acc = main()
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{acc}\n")
    i = i+1

    if acc > best_acc:
        best_acc = acc
        best_config = {
            'n_tree': n_tree,
            'tree_depth': t_depth,
            'batch_size': batch_size,
            'hidden_dim': hd,
            'tree_feature_rate': feature_rate,
            'feat_dropout': dropout,
            'lr': lr
        }

print("\nBest hyperparameter configuration:")
print(best_config)
print(f"Best accuracy: {best_acc}")



Running: n_tree=10, t_depth=7, hd=768, batch_size=512, feature_rate=0.1, dropout=0.1, lr=0.01
1 / 100
Use gtd200 dataset
Patience: 100


Training Epochs:  61%|██████▏   | 245/400 [00:29<00:18,  8.33it/s]


Early stopping at epoch 246

Best Accuracy: 0.502381

Running: n_tree=100, t_depth=10, hd=768, batch_size=512, feature_rate=0.1, dropout=0.0, lr=0.01
2 / 100
Use gtd200 dataset
Patience: 100


Training Epochs:  70%|███████   | 281/400 [05:15<02:13,  1.12s/it]


Early stopping at epoch 282

Best Accuracy: 0.515476

Running: n_tree=20, t_depth=8, hd=1024, batch_size=256, feature_rate=0.1, dropout=0.3, lr=0.01
3 / 100
Use gtd200 dataset
Patience: 100


...

Training Epochs:  73%|███████▎  | 293/400 [01:24<00:30,  3.48it/s]

Early stopping at epoch 294

Best Accuracy: 0.513095

Running: n_tree=100, t_depth=8, hd=1024, batch_size=256, feature_rate=0.3, dropout=0.1, lr=0.001
50 / 100
Use gtd200 dataset


Patience: 100


Training Epochs: 100%|██████████| 400/400 [12:21<00:00,  1.85s/it]



Best Accuracy: 0.508333

Best hyperparameter configuration:
{'n_tree': 20, 'tree_depth': 8, 'batch_size': 256, 'hidden_dim': 1024, 'tree_feature_rate': 0.1, 'feat_dropout': 0.3, 'lr': 0.01}
Best accuracy: 0.5369047619047619


In [9]:
#Running: n_tree=100, t_depth=10, hd=1024, batch_size=512, feature_rate=0.3, dropout=0.1, lr=0.01



In [10]:

"""
========== Final Test Evaluation ==========
Model Parameters:
  Dataset: gtd200
  Hidden Dim: 768
  n_tree: 100, tree_depth: 10, tree_feature_rate: 0.1
  Batch size: 512, Dropout: 0.1, LR: 0.01

Best Accuracy: 0.9011
Weighted Precision: 0.9030, Recall: 0.9011, F1 Score: 0.8994, ROCAUC: 0.9971
Macro Precision: 0.9030, Recall: 0.9011, F1 Score: 0.8994, ROCAUC: 0.9971
Micro Precision: 0.9011, Recall: 0.9011, F1 Score: 0.9011, ROCAUC: 0.9976
"""

'\n========== Final Test Evaluation ==========\nModel Parameters:\n  Dataset: gtd200\n  Hidden Dim: 768\n  n_tree: 100, tree_depth: 10, tree_feature_rate: 0.1\n  Batch size: 512, Dropout: 0.1, LR: 0.01\n\nBest Accuracy: 0.9011\nWeighted Precision: 0.9030, Recall: 0.9011, F1 Score: 0.8994, ROCAUC: 0.9971\nMacro Precision: 0.9030, Recall: 0.9011, F1 Score: 0.8994, ROCAUC: 0.9971\nMicro Precision: 0.9011, Recall: 0.9011, F1 Score: 0.9011, ROCAUC: 0.9976\n'

In [11]:
sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(best_config['n_tree']),
        '-tree_depth', str(best_config['tree_depth']),
        '-batch_size', str(best_config['batch_size']),
        '-hidden_dim', str(best_config['hidden_dim']),
        '-epochs', '1500',
        '-verbose', '0',
        '-tree_feature_rate', str(best_config['tree_feature_rate']),
        '-feat_dropout', str(best_config['feat_dropout']),
        '-lr', str(best_config['lr']),
        '-jointly_training',
        '-searching', '0'
    ]

best_model, preds, targets, labels, epoch_logs = main()

Use gtd200 dataset


Patience: 300


Training Epochs:   3%|▎         | 50/1500 [00:22<10:43,  2.25it/s]

[Epoch 50] Train Loss: 1.4580, Eval Loss: 1.9062, Eval Accuracy: 0.4583


Training Epochs:   7%|▋         | 100/1500 [00:45<11:11,  2.08it/s]

...

[Epoch 600] Train Loss: 1.2063, Eval Loss: 2.1378, Eval Accuracy: 0.5083


Training Epochs:  43%|████▎     | 650/1500 [04:37<05:42,  2.48it/s]

[Epoch 650] Train Loss: 1.2237, Eval Loss: 2.1436, Eval Accuracy: 0.5083


Training Epochs:  46%|████▌     | 687/1500 [04:54<05:48,  2.34it/s]

Early stopping at epoch 688
Evaluating on test set with best model...


In [12]:
from sklearn.metrics import classification_report

print(classification_report(targets, preds))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.28      0.42      0.34        60
        African National Congress (South Africa)       0.47      0.67      0.55        60
                                Al-Qaida in Iraq       0.37      0.50      0.43        60
        Al-Qaida in the Arabian Peninsula (AQAP)       0.25      0.28      0.27        60
                                      Al-Shabaab       0.35      0.20      0.26        60
             Basque Fatherland and Freedom (ETA)       0.54      0.80      0.64        60
                                      Boko Haram       0.35      0.37      0.36        60
  Communist Party of India - Maoist (CPI-Maoist)       0.44      0.60      0.51        60
       Corsican National Liberation Front (FLNC)       0.52      0.73      0.61        60
                       Donetsk People's Republic       0.35      0.45      0.39        60
Farabundo

In [13]:
def plot_confusion_matrix(y_true, y_pred, labels, partition):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix (Partition gtd{partition})", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = f"results/confusion_matrix_partition_gtd{partition}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition gtd{partition} to {save_path}")



In [14]:
plot_confusion_matrix(targets, preds, labels, partition)

ValueError: At least one label specified must be in y_true